# BM25 по заголовкам: ограничение по локации

Сравнение на одинаковых 100 контекстах test: поиск по всему корпусу и поиск в отдельном корпусе с `item_location_id == search_location_id`. Используем только `train.parquet`.

Без справочника географии это проверка точного совпадения ID локации, а не иерархии регионов. Для каждого локального корпуса строится свой BM25: меняются и кандидаты, и корпусные статистики (IDF, средняя длина заголовка).

Recall усредняем по всем запросам, без выдачи — 0. Целевые пары при фильтрации не удаляем. Отдельно считаем верхнюю границу Recall при таком ограничении.

In [1]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/split.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.split import split_queries
from src.retrievers.bm25 import BM25Retriever

SEED = 42
N_TEST_QUERIES = 100
KS = [10, 50, 100]
QUERY_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]

## Данные, сплит и общие целевые объявления

In [2]:
data = pl.read_parquet(
    ROOT / "data/train.parquet",
    columns=QUERY_COLUMNS + ["item_id", "item_title_raw", "item_location_id"],
)
data = data.with_columns(pl.Series("split", split_queries(data["search_query"], seed=SEED)))
items = pd.DataFrame(
    data.select("item_id", "item_title_raw", "item_location_id")
    .unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").to_dict(as_series=False)
)
test = (
    data.filter(pl.col("split") == "test")
    .group_by(QUERY_COLUMNS).agg(pl.col("item_id").unique().alias("targets"))
    .sort(QUERY_COLUMNS)
)
test = test.sample(n=min(N_TEST_QUERIES, len(test)), seed=SEED, shuffle=True)
queries = pd.DataFrame(test.select(QUERY_COLUMNS).to_dict(as_series=False))
targets = [set(ids) for ids in test["targets"].to_list()]
assert all(targets)
assert set().union(*targets).issubset(set(items["item_id"]))
print(f"Корпус: {len(items):,}; запросов: {len(queries)}; локаций: {queries.search_location_id.nunique()}")

Корпус: 344,825; запросов: 100; локаций: 70


## Recall@K

Самописная функция возвращает Recall каждого контекста; среднее берётся по всей тестовой выборке. Знаменатель — число уникальных наблюдаемых объявлений до фильтрации.

In [3]:
def recall_per_query(candidates, targets, k):
    retrieved = (
        candidates.loc[candidates["rank"] <= k]
        .groupby("query_index")["item_id"].agg(set).to_dict()
    )
    return np.array([
        len(relevant & retrieved.get(i, set())) / len(relevant)
        for i, relevant in enumerate(targets)
    ])

## Без ограничения локации

In [4]:
model = BM25Retriever(lemmatize=False, batch_size=16)
model.fit(items[["item_id", "item_title_raw"]])
global_candidates = model.retrieve(queries, k=max(KS))
del model
gc.collect();

## Корпус по локации запроса

Индексируем все доступные объявления каждой выбранной локации. После поиска переводим локальные позиции запросов в позиции общей таблицы `queries`. Если корпус пуст, выдача остаётся пустой без подстановки глобальных кандидатов.

In [5]:
parts = []
corpus_sizes = np.zeros(len(queries), dtype=int)
reachable_recall = np.zeros(len(queries))

for location, query_group in queries.groupby("search_location_id", sort=True):
    positions = query_group.index.to_numpy()
    corpus = items.loc[items["item_location_id"] == location, ["item_id", "item_title_raw"]]
    available = set(corpus["item_id"])
    corpus_sizes[positions] = len(corpus)
    reachable_recall[positions] = [
        len(targets[i] & available) / len(targets[i]) for i in positions
    ]
    if corpus.empty:
        continue
    model = BM25Retriever(lemmatize=False, batch_size=16)
    model.fit(corpus)
    candidates = model.retrieve(query_group, k=max(KS))
    if not candidates.empty:
        candidates["query_index"] = positions[candidates["query_index"].to_numpy(dtype=int)]
        parts.append(candidates)
    del model
    gc.collect()

local_candidates = (
    pd.concat(parts, ignore_index=True).sort_values(["query_index", "rank"]).reset_index(drop=True)
    if parts else pd.DataFrame(columns=["query_index", "item_id", "rank", "score"])
)
# Проверяем именно состав выдачи, а не только способ формирования корпуса.
item_locations = items.set_index("item_id")["item_location_id"]
assert all(
    item_locations[item_id] == queries.iloc[int(i)]["search_location_id"]
    for i, item_id in local_candidates[["query_index", "item_id"]].itertuples(index=False, name=None)
)
assert not local_candidates.duplicated(["query_index", "item_id"]).any()

## Сравнение

In [6]:
comparison = pd.DataFrame({
    "Без фильтра": [recall_per_query(global_candidates, targets, k).mean() for k in KS],
    "Совпадение локации": [recall_per_query(local_candidates, targets, k).mean() for k in KS],
}, index=[f"Recall@{k}" for k in KS])
comparison["Разница"] = comparison["Совпадение локации"] - comparison["Без фильтра"]
display(comparison)
print(f"Верхняя граница Recall при фильтре: {reachable_recall.mean():.4f}")
print(f"Медианный локальный корпус: {np.median(corpus_sizes):,.0f} объявлений")
print(f"Запросов с пустым корпусом: {(corpus_sizes == 0).sum()}")
print(f"Запросов без локальных кандидатов: {len(queries) - local_candidates.query_index.nunique()}")

,Без фильтра,Совпадение локации,Разница
Recall@10,0.050,0.383333,0.333333
Recall@50,0.085,0.455000,0.370000
Recall@100,0.155,0.478333,0.323333


Верхняя граница Recall при фильтре: 0.7900
Медианный локальный корпус: 1,746 объявлений
Запросов с пустым корпусом: 13
Запросов без локальных кандидатов: 23


## Примеры изменения Recall@10

In [7]:
details = queries[["search_query", "search_location_id"]].copy()
details["corpus_size"] = corpus_sizes
details["reachable_recall"] = reachable_recall
details["recall_global"] = recall_per_query(global_candidates, targets, 10)
details["recall_local"] = recall_per_query(local_candidates, targets, 10)
details["difference"] = details["recall_local"] - details["recall_global"]
display(details.sort_values("difference", ascending=False).head(5))
display(details.sort_values("difference").head(5))

,search_query,search_location_id,corpus_size,reachable_recall,recall_global,recall_local,difference
1,лидогенерация,635320,4648,1.0,0.0,1.0,1.0
2,чистовая отделка квартир,622660,807,1.0,0.0,1.0,1.0
5,попутный эвакуатор,659240,178,1.0,0.0,1.0,1.0
4,сауна снять,640860,5620,1.0,0.0,1.0,1.0
28,переварка ваз,644200,3817,1.0,0.0,1.0,1.0


,search_query,search_location_id,corpus_size,reachable_recall,recall_global,recall_local,difference
0,сколы на лобовом,623410,0,0.0,0.0,0.0,0.0
3,ремонт компьютеров и ноутбуков,652100,1011,1.0,0.0,0.0,0.0
6,в твери подкат арендовать,656901,2011,1.0,0.0,0.0,0.0
11,строительная бригада,641040,6,1.0,0.0,0.0,0.0
12,ремонт гбц,641780,5862,1.0,0.0,0.0,0.0
